In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
"""
@article{lecun2010mnist,
  title={MNIST handwritten digit database},
  author={LeCun, Yann and Cortes, Corinna and Burges, CJ},
  journal={ATT Labs [Online]. Available: http://yann.lecun.com/exdb/mnist},
  volume={2},
  year={2010}
}
"""
from datasets import load_dataset

ds = load_dataset("ylecun/mnist")
def get_mnist_datasets():
    train_ds = ds["train"]
    test_ds = ds["test"]
    return train_ds, test_ds

train_ds, test_ds = get_mnist_datasets()

print(f"Number of training samples: {len(train_ds)}")
print(f"Number of test samples: {len(test_ds)}")

Number of training samples: 60000
Number of test samples: 10000


In [4]:
def learn(data, weights):
    # returns the dot product of weights with input units to produce a feature
    return np.dot(data, weights)

In [5]:
def generate_random_weights(input_count, neuron_count):
    """
    neuron_count: number of neurons in hidden layer - in our case 128
    input_count: number of inputs which we initally started with - 784
    returns random weights 128x784 
    Weights from theoritically from -inf to +inf but this would produce absurd values to learn. 
    Guassian distribution - mean = 0, sd = 0.1, this ensures 68% values between -0.1 to 0.1, 99.7% between -0.3 to 0.3
    Smaller weights = easier to tune/ sensitive
    """
    weights = np.random.normal(loc=0.0, scale=0.0001, size=(neuron_count, input_count))
    np.save('weights.npy', weights)
    return(weights)

In [6]:
def activation_function(feature):
    #ReLU -> -ve weighted features are converted to 0
    return(max(0, feature))

In [7]:
def hidden_unit(data, weights):
    # Singular unit from hidden layer. Represents a feature of dataset
    unit = activation_function(learn(data, weights))
    return unit

In [8]:
def softmax(features):
    updated_features = []
    for feature in features:
        updated_features.append(np.e ** feature)

    total_sum = np.sum(updated_features)
    return [i / total_sum for i in updated_features]

In [9]:
def error_l1(w2, err_l2, h):
    e = np.dot(w2.T, err_l2)
    return(np.sign(h) * e)

In [10]:
def error_l2(p, l):
    e = []
    
    for i in range(len(p)):
        if i == l:
            e.append(1-p[i])    
        else:
            e.append(-p[i])

    return np.array(e)

In [11]:
def cost(layer, err):
    #print("Hidden Layer", layer)
    return err[:, None] * layer

In [12]:
def update_weights(cost, weights, lr=0.01):
    return weights + cost * lr

In [13]:
def train(training_dataset):
    layer1_weights = np.array(generate_random_weights(784, 128))
    layer2_weights = np.array(generate_random_weights(128, 10))
    
    print("Layer 2 Weights", layer2_weights)
    
    for sample in training_dataset.select(range(60000)):
        image = sample["image"]
        label = sample["label"]
        data = np.array(list(image.getdata()))
        data = data / 255.0
        
        layer1 = []
        for w in layer1_weights:
            unit = hidden_unit(data, w)
            layer1.append(unit)

        #print("Layer1",layer1)
        #print(len(layer1))
        
        layer2 = []
        for w in layer2_weights:
            unit = learn(layer1, w)
            layer2.append(unit)

        #print("Layer2",layer2)
        #print(len(layer2))
        
        prediction = softmax(layer2)

        #print("Softmax:", prediction)
        
        err_l2 = np.array(error_l2(prediction, label))
        
        err_l1 = error_l1(layer2_weights, err_l2, layer1)

        
        loss_l2 = cost(layer1, err_l2)

        loss_l1 = cost(data, err_l1)

        #print("Cost", loss)
        #print("L1 cost", loss_l1)

        layer2_weights = update_weights(loss_l2, layer2_weights)
        layer1_weights = update_weights(loss_l1, layer1_weights)

    print("Final layer 2 weights:", layer2_weights)
    return({'w1':layer1_weights, 'w2':layer2_weights})
    #print("accuracy:", np.sum(predicted)/len(predicted)*100)

In [ ]:
training = train(train_ds)

Layer 2 Weights [[ 1.14448304e-04 -2.02052909e-04  5.26084614e-05 ... -7.90934712e-05
  -5.93192842e-06 -8.53345764e-05]
 [-1.98372348e-05 -4.20381218e-05  9.42361534e-05 ... -1.24055312e-04
  -3.88658449e-05 -1.31995341e-04]
 [ 1.13800194e-04 -1.65057055e-04  4.92470280e-07 ... -1.47550887e-04
  -4.92301182e-05  2.02279266e-04]
 ...
 [-1.28267320e-05 -1.46100825e-04 -1.26181887e-04 ... -7.61924745e-05
  -1.67678292e-05  1.59833856e-05]
 [-3.10323044e-05 -4.95321803e-05  1.74935406e-04 ... -1.05970789e-04
   5.60447447e-05  3.07234330e-05]
 [-6.51575305e-05 -3.50770044e-05  6.70835873e-05 ... -3.86414715e-05
  -4.13588216e-05  3.16554208e-05]]


In [143]:
def test(test_dataset, w1, w2):
    predicted = []

    for sample in test_dataset.select(range(len(test_ds))):
        image = sample["image"]
        label = sample["label"]
        data = np.array(list(image.getdata()))
        data = data / 255.0
        
        l1 = []
        for w in w1:
            unit = hidden_unit(data, w)
            l1.append(unit)

        l2 = []
        for w in w2:
            unit = learn(l1, w)
            l2.append(unit)

        #print(l2)
        
        p = softmax(l2)

        #print("Softmax Output", p)
        
        p_l = np.argmax(p)
        if p_l == label:
            predicted.append(1)
        else:
            predicted.append(0)

        #print("Predicted vs Label:", p_l, label)

    print(len(predicted))
    print("Accuracy(%): ", np.sum(predicted)/10000*100)

In [144]:
test(test_ds, training['w1'], training['w2'])

10000
Accuracy(%):  95.3


In [149]:
def run(image, w1, w2):
    data = np.array(list(image.getdata()))
    data = data / 255.0
    
    l1 = []
    for w in w1:
        unit = hidden_unit(data, w)
        l1.append(unit)

    l2 = []
    for w in w2:
        unit = learn(l1, w)
        l2.append(unit)
    
    p = softmax(l2)

    #print("Softmax Output", p)
    
    p_l = np.argmax(p)
    print(p_l)


from PIL import Image

img = Image.open("/mnt/c/Users/Administrator/Desktop/80_Days_ML/week1/test_2.png")
img.show()
run(img.convert("L"), training['w1'], training['w2'])

8


In [ ]:
import json

file_path = "weights.json"

data = {k: v.tolist() for k, v in training.items()}
np.save('weights.npy', training)

with open(file_path, "w") as json_file:
     json.dump(data, json_file, indent=4)